# Table of contents

- **Introduction**
- **Objectives**
- **Project Overview**
- **Deep Q-Network (DQN) Algorithm Foundations**
- **Deep Q-Network with Keras implementation with Keras**
    * Environment Setup
    * Deep Q-Network (DQN) Setup
    * Replay Buffer
    * The epsilon-greedy policy
    * Deep Q-Learning Operational Loop
    * Deep Q-Learning Training Loop and Agent-Environment Interaction
    * Evaluate the performance
- **Experiments**
    * Experiment 1: Explicit Reward Shaping for Stability
    * Experiment 2: Early Stopping for Efficiency
    * Experiment 3: Adaptive Greedy $\epsilon$-Decay Strategy
- **Experiments Summary**

# Introduction

This hands-on tutorial notebook documents the step-by-step implementation of a Deep Q-Network (DQN) using the Keras framework. DQN is a key algorithm in Reinforcement Learning (RL) that effectively combines neural networks with classic Q-Learning, enabling agents to solve environments with complex, high-dimensional state spaces.

The primary goal of this project is the training of an agent to master the CartPole-v1 environment. This is a classic control problem requiring the agent to balance a pole on a moving cart for the maximum possible duration. The completion of this tutorial provides knowledge on how to apply DQNs in Keras to define and optimize action-selection policies for a given environment.

# Objectives

**Tutorial Roadmap: The Complete DQN Pipeline**

This notebook systematically breaks down the entire DQN pipeline, focusing on step-by-step implementation and understanding how core design choices influence the learning process. The following key areas are covered:

**1. Fundamental Setup**

- Environment Initialization: Setting up the CartPole-v1 environment and configuring random seeds to ensure reproducible results.
- DQN Model Definition: Defining the Keras neural network architecture (the Q-Network) used to approximate the optimal action-value function, $Q ∗ (s,a)$.

**2. Training Mechanism**

- Experience Replay: Implementation of a memory buffer to store and sample past transitions, which is crucial for stabilizing the training process.
- ϵ-Greedy Strategy: Implementation of this mechanism to dynamically manage the critical exploration-exploitation trade-off during training.

**3. Evaluation and Analysis**

- Agent Evaluation: Assessment of the final performance of the trained DQN agent.
- Experimental Insights: Beyond the core implementation, targeted experiments are conducted to demonstrate the effects of:
- Network Capacity: Testing the impact of a larger architecture (e.g., 3×64 units).
- Adaptive Exploration: Modifying the ϵ decay schedule based on agent performance.
- Reward Shaping: Providing dense, continuous state feedback instead of sparse rewards.

**Tutorial Objective**

Upon completion of this tutorial, the following skills will be attained:

- Implementation of a Deep Q-Network using Keras.
- Definition and training of a neural network to approximate the Q-values.
- Evaluation of the performance of the trained DQN agent.

# Project Overview

This academic project implements and analyzes a complete Deep Q-Learning pipeline, focusing on training an agent to solve the classic CartPole-v1 control problem. The project’s structure is modular, addressing environment configuration, model construction, hyperparameter tuning, and advanced experimental analysis.

**I. Methodology and Core Components**

The pipeline is built on the fundamental components of the DQN algorithm, ensuring stability and reproducibility:

- **Environment Setup:** The CartPole-v1 environment (OpenAI Gym) is initialized with 4 continuous state variables (cart position, cart velocity, pole angle, pole angular velocity) and 2 discrete actions (left/right). Random seeds (NumPy, TensorFlow, Gym) are set to guarantee reproducible results.
- **Neural Network Model:** A Deep Q-Network (DQN) is constructed using Keras/TensorFlow to approximate the optimal action-value function, \( Q^*(s, a) \). The architecture consists of two hidden layers (24 neurons each) with ReLU activation and a linear output layer (2 neurons). The model is compiled with MSE loss and the Adam optimizer (learning rate 0.001).
- **Training Loop & Stability:** Training relies on Experience Replay (deque buffer of size 2000) to store transitions \( (s, a, r, s', done) \). Action selection uses an ϵ-Greedy Policy with exponential decay (ϵ starts at 1.0, decays by 0.995 per step, min 0.01). A separate Target Network is used for stable Q-target calculation, with hard updates every 10 episodes. The discount factor γ is set to 0.95.

**II. Targeted Experimental Analysis**

The project includes specific experiments designed to quantify the impact of key design choices on learning performance:

- **Reward Shaping for Stability:** A custom reward function penalizes large pole angles (penalty proportional to angle) to incentivize the agent to keep the pole more vertical. Results showed high variance and slow convergence, suggesting the penalty may have been too disruptive within the limited 50-episode training window.
- **Early Stopping for Efficiency:** An early stopping criterion halts training when the agent consistently achieves scores ≥195 for 100 consecutive episodes. While the criterion was not triggered within 50 episodes, the agent demonstrated near-perfect scores (199) by episode 33, proving the core model’s capability and the potential for computational savings in longer runs.
- **Adaptive ϵ-Decay Strategy:** A hybrid decay schedule was implemented: linear decay (step size 0.01) for the first 25 episodes to encourage broad exploration, followed by exponential decay (multiplier 0.995) to rapidly shift to exploitation. This strategy achieved perfect scores (199) by episode 4 and maintained them for the remainder of training, demonstrating the strongest efficiency and convergence.

**III. Robustness and Evaluation**

The pipeline incorporates robust error and warning handling to ensure compatibility across various Gym API versions (handling both 4-tuple and 5-tuple step returns) and to mitigate common environment-related issues like rendering errors in headless environments.

The agent’s performance is rigorously evaluated using a fully greedy policy (ϵ=0) over 10 test episodes, yielding metrics such as average reward (14.60), max reward (65.0), and min reward (8.0) for the baseline run. The high variance observed underscores the need for the targeted experiments to improve consistency. The final adaptive ϵ-decay experiment produced a stable, optimal policy with a score of 199 in all subsequent episodes, validating the effectiveness of dynamic exploration scheduling.

# Deep Q-Network (DQN) Algorithm Foundations

The foundation of the notebook is built upon the standard RL theories and concepts, which involves an Agent interacting with an Environment. The specific environment used is CartPole-v1 from the Gym toolkit, a classic control problem where the goal is to prevent a pole from falling over . Key components of this interaction include the State or Observation Space (the input to the agent, describing the current situation), the discrete Action Space (the available moves the agent can take), and the Reward (the scalar feedback received after each action). The process of learning occurs over multiple Episodes (complete runs from start to termination). The Discount Factor (γ) is a hyperparameter used to weigh the present value of future rewards. Finally, the use of Random Seeds ensures the experiment is reproducible.

**Deep Q-Network (DQN) Algorithm**

The central theoretical concept is the Deep Q-Network (DQN), an algorithm that combines Q-Learning (a temporal difference control algorithm) with neural networks to handle environments with large state spaces.

- **Q-Value $(Q(s,a))$:** The value being estimated, representing the expected cumulative discounted future reward for taking action a in state s.
- **Function Approximation:** A neural network, referred to as the Q-Network, is used to approximate the optimal Q-values, $Q(s,a)$, replacing the traditional, unstable Q-table.
- **Q-Target / Bellman Equation:** The Q-learning update (implemented in the `replay()` function) uses the Bellman Equation to define the target value: the sum of the immediate reward and the discounted maximum Q-value of the next state. This target is used as the ground truth for training the network.

**Deep Learning and Keras Implementation**

The Q-Network is structured using Keras and relies on standard deep learning concepts:

- **Sequential Model:** The network architecture is defined as a linear stack of layers.
- **Dense Layers:** Fully connected layers form the body of the network. The input layer's dimension corresponds to the state size.
- **Activation Functions:** The Rectified Linear Unit (ReLU) is used in the hidden layers for non-linearity, while a Linear activation is used in the output layer since the Q-values are continuous.
- **Loss Function (MSE):** The Mean Squared Error is used to measure the difference between the network's predicted Q-values and the calculated Q-Targets.
- **Optimizer (Adam):** The Adam optimization algorithm is employed to perform Backpropagation and adjust the network's weights to minimize the loss.

**Training Stability and Control Mechanisms**

To stabilize the learning process, the DQN implements two critical techniques:

- **Experience Replay:** This technique involves storing the agent's experiences (state, action, reward, next state, done tuples) in a Replay Buffer (implemented with a deque). Training is performed on a randomly sampled Minibatch of these experiences. This process breaks the temporal correlation between sequential samples, leading to more stable and efficient learning.
- **ϵ-Greedy Policy:** This strategy is used to manage the Exploration-Exploitation Trade-off. With a probability of ϵ (the Exploration Rate), the agent takes a random action (Exploration); otherwise, it takes the action with the maximum predicted Q-value (Exploitation). The value of ϵ is subjected to ϵ-Decay, gradually decreasing over time to transition the agent from purely exploring to primarily exploiting its learned knowledge.

**Advanced Techniques and Evaluation**

The final sections introduce advanced concepts essential for optimizing RL agents:

- **Reward Shaping :** The practice of augmenting the environment's default reward structure to provide better gradient information, effectively guiding the agent toward the desired long-term goal.
- **Early Stopping :** A form of regularization and efficiency control that automatically halts the training loop once the agent consistently achieves a satisfactory performance threshold, preventing wasted computation.
- **Adaptive Exploration :** Implementing complex decay schedules (e.g., switching between linear and exponential decay) for ϵ to fine-tune the exploration strategy across different phases of the learning process.
- **Hyperparameters :** The overall learning process is governed by a set of Hyperparameters, including γ, batch_size, learning_rate, ϵ, and epsilon_decay.

# Deep Q-Network with Keras implementation with Keras

## Environment Setup

The experimental foundation is established using the highly recognized CartPole-v1 environment from the OpenAI Gym suite, which serves as a foundational benchmark problem within the field of Reinforcement Learning (RL).

The CartPole problem is formally defined as a Markov Decision Process (MDP), which is essential for applying standard RL algorithms. The agent's core task is one of stabilization and control: to prevent the pole from falling beyond a critical angle (15 degrees) by applying discrete, lateral forces to the cart.

The environment's components map directly to RL concepts:

- **State Space ($\mathcal{S}$)**: The agent observes a four-dimensional continuous state space, comprising the cart's position, cart's velocity, pole's angle, and pole's angular velocity. This state provides the agent with the necessary information (Markov Property) to make optimal decisions.
- **Action Space ($\mathcal{A}$)**: The agent operates within a discrete action space, where it can only apply a constant force to the left or to the right. This binary choice makes it a suitable test case for Q-learning or Policy Gradient methods with discrete outputs.
- **Reward Function ($R$)**: The agent receives a reward of +1 for every single time step the pole remains balanced. The overall objective is to maximize the cumulative future reward (or return), reinforcing the stability behavior over time and requiring the agent to develop a long-term policy ($\pi$).

This setup models an episodic task, where failure (the pole falling or the cart moving out of bounds) terminates the episode, forcing the agent to learn complex control sequences to achieve maximum performance (typically a 500-step balance).

**Packages Installation**

In [ ]:
!pip install gym==0.25.2  # Specific version for compatibility

**Import libraries**

In [ ]:
import gym
import numpy as np
import tensorflow as tf
import warnings
import os

2025-10-18 14:01:52.217476: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760796112.494516      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760796112.573332      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


**Configurations**

In [ ]:
# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=DeprecationWarning)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # Suppress TensorFlow logs
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Force CPU execution

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
        
# Create the environment
env = gym.make('CartPole-v1', new_step_api=True, render_mode='rgb_array')
env.action_space.seed(42)

# Define state and action sizes
state_size = env.observation_space.shape[0]  # 4 for CartPole (position, velocity, angle, angular velocity)
action_size = env.action_space.n             # 2 for CartPole (left, right)

## Deep Q-Network (DQN) Setup

The agent's policy learning is implemented through a Deep Q-Network (DQN) architecture, which merges the principles of Q-learning with deep neural networks.

Instead of maintaining a massive look-up table for the Q-values, the DQN utilizes a Keras-defined neural network as a powerful, non-linear function approximator to estimate the action-value function, denoted as $Q(s, a)$. The network takes the current state ($s$) as input and outputs a Q-value for every possible discrete action ($a$). 

The primary goal of the network is to learn $Q^*(s, a)$, the maximum expected return (discounted cumulative reward) achievable by performing action $a$ in state $s$ and subsequently following the optimal policy. The training process minimizes the Temporal Difference (TD) Error, which measures the difference between the network's current Q-value estimate and a more stable target Q-value derived from the Bellman Equation for optimality.

To ensure stable convergence in this non-stationary environment, the DQN implementation incorporates two critical stabilizing techniques:

- **Experience Replay Buffer:** Past state-transition tuples ($s_t, a_t, r_{t+1}, s_{t+1}$) are stored in a memory buffer and sampled randomly during training. This breaks the temporal correlations in the sequential data and smooths the learning process.
- **Target Network:** A separate, delayed copy of the primary Q-network (the Target Network) is used to calculate the stable target Q-values for the TD Error. This separation prevents the network from chasing a constantly moving target, dramatically improving stability and convergence.

**DQN Model Architecture and Optimization**

The Keras sequential model defines the mapping from the observed state to the predicted Q-values.

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-k9u1{border-color:inherit;color:#1B1C1D;font-size:100%;text-align:left;vertical-align:bottom}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-k9u1">Component</th>
    <th class="tg-7zrl">Architecture Detail</th>
    <th class="tg-7zrl">RL/DL Concept</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-7zrl">Input Layer</td>
    <td class="tg-7zrl">4 neurons (matching the dimension of $\mathcal{S}$)</td>
    <td class="tg-0lax">State Representation: Directly consumes the continuous state vector (position, velocity, angle, angular velocity).</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Hidden Layers</td>
    <td class="tg-7zrl">Two layers, each with 24 neurons</td>
    <td class="tg-0lax">Feature Extraction: Provides the capacity for the network to extract complex, non-linear relationships from the input state features.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Activation</td>
    <td class="tg-7zrl">Rectified Linear Unit (ReLU)</td>
    <td class="tg-0lax">Non-Linearity: Crucial for allowing the neural network to approximate complex functions, enabling it to learn control policies beyond simple linear mappings.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Output Layer</td>
    <td class="tg-7zrl">2 neurons (matching the dimension of $\mathcal{A}$)</td>
    <td class="tg-0lax">Action-Value Prediction: Outputs the estimated Q-value for each possible action (Left or Right).</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Output Activation</td>
    <td class="tg-7zrl">Linear</td>
    <td class="tg-0lax">Ensures the output Q-values can take on any necessary positive or negative magnitude without being bounded by an activation function.</td>
  </tr>
</tbody></table>

**Optimization Strategy:**

The training objective is to minimize the Temporal Difference (TD) Error via a robust optimization scheme:

- **Loss Function:** The Mean Squared Error (MSE) is employed as the loss function. This function quantifies the distance between the predicted Q-value ($Q(s_t, a_t)$) and the target Q-value ($Y_t$), directly minimizing the TD Error:


$$\text{Loss} = \frac{1}{N} \sum_{i=1}^{N} (Y_i - Q(s_i, a_i))^2$$

- **Optimizer:** The Adam (Adaptive Moment Estimation) optimizer is used with a fixed learning rate ($\alpha$) of 0.001. Adam is chosen for its efficiency and adaptive control over the learning rate for each network parameter, which accelerates convergence compared to standard Stochastic Gradient Descent.

- **Target Update Mechanism:** The stable Q-value estimates provided by the Target Network are achieved by periodically performing a "hard update," where the weights of the primary prediction network are copied directly to the Target Network at a set interval. This delay is critical for stabilizing the iterative self-improvement required by Q-learning.

In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

def build_model(state_size, action_size):
    """Constructs a Deep Q-Network (DQN) model."""
    model = Sequential([
        Input(shape=(state_size,)),
        Dense(24, activation='relu'),  # Hidden layer 1
        Dense(24, activation='relu'),  # Hidden layer 2
        Dense(action_size, activation='linear')  # Output layer for Q-values
    ])
    model.compile(loss='mse', optimizer=Adam(learning_rate=0.001))
    return model

# Initialize main and target models
model = build_model(state_size, action_size)
target_model = build_model(state_size, action_size)
target_model.set_weights(model.get_weights())  # Sync weights initially

2025-10-18 14:02:10.826302: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Replay Buffer

The Replay Buffer is a crucial component for stabilizing the training of the DQN agent, effectively transitioning the algorithm into an off-policy method. It addresses the fundamental problem of highly correlated data, where sequential samples in an episode are statistically dependent, which violates the assumptions of independent and identically distributed (i.i.d.) data required for effective stochastic gradient descent.

**Buffer Mechanics**

- **Data Structure:** The buffer is implemented as a deque (double-ended queue), which acts as a circular buffer with a defined maximum capacity (set to 2000 in this implementation). When the buffer is full, the oldest experience tuple is automatically discarded to make room for the newest one, ensuring the buffer always holds a diverse, but recently collected, set of experiences.

- **Experience Tuple:** Each entry stored in the buffer is a complete state transition tuple defined as $(s_t, a_t, r_{t+1}, s_{t+1}, \text{done})$.

    - $s_t$: The current state observed.
    - $a_t$: The action taken by the agent.
    - $r_{t+1}$: The reward received.
    - $s_{t+1}$: The next state resulting from the transition.
    - $\text{done}$: A boolean flag indicating if the episode terminated.

By sampling random mini-batches from this buffer, the DQN algorithm decorrelates the data, which reduces the variance of the updates. This reuse of past experiences is a key characteristic of off-policy learning, where the agent learns the optimal Q-function ($Q^*$) based on data collected by a separate, often exploratory, behavioral policy.

In [4]:
from collections import deque
import random

# Initialize replay buffer
memory = deque(maxlen=2000)

def remember(state, action, reward, next_state, done):
    """Stores an experience in the replay buffer."""
    memory.append((state, action, reward, next_state, done))

Meaning of `maxlen` parameter

The `maxlen` parameter defines the maximum capacity of the deque object, which is used as the agent's replay buffer (or memory).

Fixed Size: The `deque` (double-ended queue) is initialized with a maximum size of 2,000. This means the buffer can hold a maximum of 2,000 past experiences (transitions).

FIFO (First-In, First-Out): When the buffer is full (i.e., it already contains 2,000 experiences) and a new experience is added using `memory.append()`, the oldest experience is automatically discarded from the left side of the queue to make room for the new experience on the right side.

Why `maxlen` is Essential in DQL
This fixed-size, revolving memory serves three critical purposes in Deep Q-Learning:

- **Experience Replay:** DQL models learn by sampling small batches of experiences randomly from this memory. Since the memory stores experiences gathered over time, this process breaks the temporal correlations in the data. If the model only learned from the last step, its training would be unstable.
- **Non-Stationarity:** By limiting the size of the buffer (e.g., to 2,000), you prevent the agent from dwelling too long on very old experiences that might have been gathered by an early, poorly performing version of the Q-network. This ensures the agent focuses on more recent, relevant data.
- **Memory Management:** Setting a maximum size prevents the replay buffer from growing indefinitely and consuming all available system memory, especially in environments where episodes run for thousands of steps.

## The epsilon-greedy policy

**The epsilon ($\epsilon$) -Greedy Policy**

The agent employs the $\epsilon$-Greedy policy as its behavioral policy to manage the critical Exploration-Exploitation Trade-off. This trade-off is fundamental to Reinforcement Learning, requiring the agent to balance investigating unknown states and actions (Exploration) with making the currently known best move (Exploitation) to maximize immediate reward.

The $\epsilon$-Greedy mechanism ensures continuous learning while guiding the agent toward optimal behavior:

- **Exploration:** With a probability of $\epsilon$ (epsilon), the agent selects an action $a_t$ uniformly at random from the entire Action Space ($\mathcal{A}$). This stochastic behavior ensures the agent continues to discover new state-action value pairs, potentially revealing better strategies.
- **Exploitation:** With the complementary probability of $(1 - \epsilon)$, the agent selects the greedy action $a_t = \text{argmax}_a Q(s_t, a)$, which is the action estimated by the Q-network to yield the highest expected future return for the current state $s_t$.

**Epsilon $\epsilon$ Decay Schedule**

To transition the agent from an initial state of high exploration to a state of high exploitation, the parameter $\epsilon$ is controlled by an annealing schedule (decaying schedule).

- **Initialization:** $\epsilon$ starts at $1.0$, guaranteeing pure exploration initially to quickly populate the Experience Replay Buffer and initialize the Q-function.
- **Decay:** $\epsilon$ decays exponentially by a decay rate of $0.995$ after every training step. This gradual reduction ensures the agent increasingly relies on its learned Q-values as training stabilizes.
- **Minimum Threshold:** $\epsilon$ is clamped at a minimum value of $0.01$. This minimum ensures the agent always retains a small degree of randomness (stochasticity) in its policy, preventing it from getting permanently stuck in a sub-optimal local minimum during later training stages.

**Computational Consistency**

The prediction logic for selecting the greedy action is explicitly constrained to run on the CPU device using `tf.device('/CPU:0')`. This is an implementation detail necessary to ensure the consistent and deterministic evaluation of the neural network's computational graph during the action selection process, contributing to the overall reproducibility of the training results.

In [5]:
# Hyperparameters for epsilon-greedy policy
epsilon = 1.0  # Initial exploration rate
epsilon_min = 0.01  # Minimum exploration rate
epsilon_decay = 0.995  # Decay rate per training step

def act(state):
    """Selects an action using the epsilon-greedy policy."""
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)  # Explore: random action
    with tf.device('/CPU:0'):
        q_values = model.predict(state, verbose=0)  # Exploit: best Q-value
    return np.argmax(q_values[0])

**Hyperparameters meaning**

These parameters define the $\epsilon$-greedy strategy, which is a core concept in Deep Q-Learning (DQL). This strategy manages the essential Exploration-Exploitation Trade-off—determining when the agent should try something new and when it should use what it has already learned.

1. epsilon (Initial Exploration Rate)

- **Meaning:** This is the initial probability that the agent will select a random action (exploration). Setting it to 1.0 ensures that at the very beginning of training, the agent explores its environment widely, gathering diverse data before it starts relying on its as-yet-uninformed Q-Network.

3. epsilon_min (Minimum Exploration Rate)

- **Meaning:** This is the floor value that epsilon can never drop below. Even after the agent has trained for a long time, setting $\epsilon_{\text{min}}$ to a small non-zero value (like 0.01) guarantees that the agent will always perform a small amount of random exploration. This prevents the agent from getting stuck in a local optimum and allows it to discover better strategies as its environment or understanding changes.

3. epsilon_decay (Decay Rate)

- **Meaning:** This is the multiplier applied to the current epsilon value after each learning step (or episode). Since this value is less than 1, it causes $\epsilon$ to gradually decrease over time. For example, if $\epsilon = 0.5$ and the decay rate is $0.995$, the new $\epsilon$ will be $0.5 \times 0.995 = 0.4975$.This decay schedule ensures the agent smoothly transitions from high exploration (to gather data) to high exploitation (to use the learned policy) as training progresses.

**The Epsilon Decay Schedule**

Collectively, these three parameters define the schedule for how the agent shifts its behavior:$$\epsilon_{\text{new}} = \max(\epsilon \times \text{epsilon\_decay}, \text{epsilon\_min})$$The agent starts at $1.0$, slowly decreases its exploration probability by $0.5\%$ each step, and stops decreasing once it hits $0.01$. This balanced schedule is critical for stable and effective DQL convergence .

## Deep Q-Learning Operational Loop

The mechanism by which the DQN agent learns is the Q-learning update rule, which is the operational form of the Bellman Optimality Equation. This equation provides the mathematical foundation for iteratively improving the agent's estimate of the optimal action-value function, $Q^*(s, a)$.

**Deriving the Temporal Difference (TD) Target**

The training process involves minimizing the Temporal Difference (TD) Error between the current predicted Q-value and a more accurate, stabilized target Q-value, known as the TD Target ($Y_t$). The update rule is applied when a mini-batch of 32 experiences is randomly sampled from the Experience Replay Buffer, which enables efficient training via Stochastic Gradient Descent (SGD) while reducing temporal correlation.

The target Q-value ($Y_t$) for non-terminal states is defined by:

$$Y_t = r_{t+1} + \gamma \max_{a'} Q_{\text{target}}(s_{t+1}, a')$$

This target represents the immediate reward ($r_{t+1}$) plus the discounted maximum expected future reward achievable from the next state ($s_{t+1}$).

- **Discount Factor ($\gamma$):** The value is set to $\gamma = 0.95$. This parameter dictates the agent's time horizon for planning. A value close to 1.0 indicates a strong emphasis on long-term rewards, which is necessary for the CartPole environment where the final goal (a long episode) requires continuous planning.
- **Target Network ($Q_{\text{target}}$):** The future Q-value term ($\max_{a'} Q_{\text{target}}(s_{t+1}, a')$) is deliberately calculated using the fixed Target Network. This critical design choice enhances the stability of the training process by ensuring the value being learned toward is consistent, preventing oscillations that would occur if the primary network were used to generate its own targets.

**Handling Terminal States**

A specific adaptation of the Q-learning rule is required for terminal states (where the $\text{done}$ flag is true). If the episode terminates (e.g., the pole falls), there are no subsequent states or future rewards. Therefore, the expected future return term is zero, and the target Q-value simplifies:

$$Y_t = r_{t+1} \quad \text{if state } s_{t+1} \text{ is terminal}$$

**Training Procedure**

The entire process involves calculating the TD Error (the difference between $Y_t$ and $Q(s_t, a_t)$) and using the Mean Squared Error (MSE) loss to update the main network's weights. After each batch update, the $\epsilon$ value is decayed to gradually shift the agent's behavioral policy from exploration to exploitation.

In [6]:
gamma = 0.95  # Discount factor
batch_size = 32  # Batch size for training

def replay(batch_size):
    """Trains the model using a batch of experiences."""
    if len(memory) < batch_size:
        return
    minibatch = random.sample(memory, batch_size)
    states = np.vstack([x[0] for x in minibatch])
    actions = np.array([x[1] for x in minibatch])
    rewards = np.array([x[2] for x in minibatch])
    next_states = np.vstack([x[3] for x in minibatch])
    dones = np.array([x[4] for x in minibatch])
    
    with tf.device('/CPU:0'):
        q_next = target_model.predict(next_states, verbose=0)  # Stable Q-values
        q_target = model.predict(states, verbose=0)  # Current Q-values
    
    for i in range(batch_size):
        target = rewards[i]
        if not dones[i]:
            target += gamma * np.amax(q_next[i])  # Bellman update
        q_target[i][actions[i]] = target
    
    model.fit(states, q_target, epochs=1, verbose=0)
    global epsilon
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

## Deep Q-Learning Training Loop and Agent-Environment Interaction

The training is structured around an iterative episodic loop that formalizes the continuous agent-environment interaction cycle . This cycle involves the agent observing a state, taking an action based on its policy, receiving a reward, and transitioning to a new state.

**Episodic Structure and Training Duration**

- **Total Episodes:** The agent is trained over a fixed duration of 50 episodes. This defines the total number of independent learning trials the agent undertakes.
- **Step Limitation:** Each episode is artificially capped at a maximum of 200 steps. This time horizon limit prevents excessively long episodes during early training and establishes the standard performance ceiling for the CartPole environment (where "solving" is generally defined by achieving an average score of 195 over 100 consecutive episodes).

**Environment API and State Management**

The code handles the slight variations in the OpenAI Gym API for starting an episode and transitioning between states.

- **Episode Start:** The environment's reset() method is used to initialize the episode. The code is designed to accept output formats from both older (4-tuple state only) and newer (5-tuple including truncation information) Gym API versions.
- **State Transition:** The environment's step() method executes the agent's chosen action and returns the critical transition information: the next state ($s'$), the immediate reward ($r$), whether the state is terminal ($\text{done}$), and whether the episode was truncated ($\text{truncated}$).

**Reward Shaping and Terminal Penalty**

While the standard environment reward is $+1$ per step, an explicit form of Reward Shaping is implemented to strongly discourage failure:

- **Failure Penalty:** If an episode terminates due to failure (i.e., the pole falls, $\text{done}$ is true), a large negative penalty of $-10$ is applied to the final reward. This negative reinforcement makes the cost of failure immediately and highly visible to the agent's Q-function calculation, encouraging it to develop a more robust, failure-avoiding policy.

**Target Network Synchronization**

To maintain the training stability enabled by the Target Network, a synchronization schedule is employed:

- **Synchronization Frequency:** The weights of the prediction network are copied to the Target Network every 10 episodes. This controlled, periodic update frequency ensures that the TD target remains consistent long enough for the prediction network to converge toward it, without allowing the target to become too stale and diverge from the latest learned policy improvements.

In [7]:
episodes = 50  # Number of training episodes
train_frequency = 5  # Train every 5 steps
target_update_frequency = 10  # Update target model every 10 episodes

for e in range(episodes):
    state = env.reset()[0] if isinstance(env.reset(), tuple) else env.reset()
    state = np.reshape(state, [1, state_size])
    for time in range(200):  # Max steps per episode
        action = act(state)
        step_result = env.step(action)
        next_state, reward, terminated, truncated, _ = step_result if len(step_result) == 5 else (*step_result, False)
        done = terminated or truncated
        reward = reward if not done else -10  # Penalty for failure
        next_state = np.reshape(next_state, [1, state_size])
        remember(state, action, reward, next_state, done)
        state = next_state
        if done:
            print(f"Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.2f}")
            break
        if time % train_frequency == 0:
            replay(batch_size)
    if (e + 1) % target_update_frequency == 0:
        target_model.set_weights(model.get_weights())  # Update target model

error: XDG_RUNTIME_DIR not set in the environment.


Episode: 1/50, Score: 43, Epsilon: 0.99
Episode: 2/50, Score: 18, Epsilon: 0.97
Episode: 3/50, Score: 31, Epsilon: 0.94
Episode: 4/50, Score: 23, Epsilon: 0.91
Episode: 5/50, Score: 34, Epsilon: 0.88
Episode: 6/50, Score: 41, Epsilon: 0.84
Episode: 7/50, Score: 35, Epsilon: 0.81
Episode: 8/50, Score: 15, Epsilon: 0.80
Episode: 9/50, Score: 17, Epsilon: 0.79
Episode: 10/50, Score: 27, Epsilon: 0.76
Episode: 11/50, Score: 31, Epsilon: 0.74
Episode: 12/50, Score: 17, Epsilon: 0.72
Episode: 13/50, Score: 11, Epsilon: 0.71
Episode: 14/50, Score: 50, Epsilon: 0.68
Episode: 15/50, Score: 60, Epsilon: 0.64
Episode: 16/50, Score: 24, Epsilon: 0.62
Episode: 17/50, Score: 42, Epsilon: 0.59
Episode: 18/50, Score: 40, Epsilon: 0.57
Episode: 19/50, Score: 13, Epsilon: 0.56
Episode: 20/50, Score: 16, Epsilon: 0.55
Episode: 21/50, Score: 21, Epsilon: 0.54
Episode: 22/50, Score: 42, Epsilon: 0.51
Episode: 23/50, Score: 78, Epsilon: 0.47
Episode: 24/50, Score: 22, Epsilon: 0.46
Episode: 25/50, Score: 50

## Evaluate the performance

The final phase of the process is the performance evaluation, which determines the effectiveness of the learned policy ($\pi$) independent of the training mechanism.

**The Greedy Evaluation Policy**

The agent switches its action selection mechanism from the exploratory $\epsilon$-Greedy policy to a Purely Greedy Policy during evaluation.

- **Action Selection:** In the evaluation phase, the exploration parameter $\epsilon$ is effectively set to zero ($\epsilon = 0$). The agent's action in every state ($s$) is always the one that maximizes the estimated Q-value, $a_t = \text{argmax}_a Q(s_t, a)$. This represents the learned deterministic policy—the best action the agent believes it has learned to take.
- **Purpose:** By eliminating randomness, the greedy policy provides a direct, unbiased measure of the quality of the Q-function approximation and its resultant control strategy.

**Metric Assessment and Test Episodes**

Performance is formally assessed over a small set of trials to determine consistency and success:

- **Evaluation Trials:** The assessment is conducted over 10 test episodes.
- **Metrics Reported:** For each episode, the primary metrics reported are the score (the number of steps survived) and the total cumulative reward (which are numerically equivalent in the `CartPole` environment unless a terminal penalty is applied). These metrics quantify the agent's ability to maximize its return.

**Operational Notes**

Rendering and Headless Environments: The explicit omission of the `env.render()` command is a necessary operational best practice in execution environments lacking a graphical display (known as "headless" environments). This avoids potential runtime errors. For visual verification of the agent's behavior, the environment's `render_mode='rgb_array'` must be utilized, allowing the visual frames to be captured and processed separately.

- **Robustness and Compatibility:** The implementation includes explicit logic for handling variable return signatures from the environment's `reset()` and `step()` methods. This ensures API compatibility across different versions of the Gym library, making the code more robust against dependency changes.
- **Environment Management:** Thorough error handling is integrated to catch unexpected issues during execution (such as rendering problems) and, critically, to ensure the environment is properly closed at the conclusion of the evaluation run, conserving system resources.

In [8]:
# Evaluation loop
evaluation_episodes = 10  # Number of evaluation episodes
scores = []  # Track scores for performance metrics
 
for e in range(evaluation_episodes):
    state = env.reset()
    if isinstance(state, tuple):  # Handle tuple output
        state = state[0]
    state = np.reshape(state, [1, state_size])
 
    total_reward = 0  # Track total reward per episode
 
    for time in range(200):  # Max steps per episode
        # Choose the greedy action
        action = np.argmax(model.predict(state)[0])
 
        # Perform action in the environment
        result = env.step(action)
        if len(result) == 4:  # Handle 4-value output
            next_state, reward, done, _ = result
        else:  # Handle 5-value output
            next_state, reward, done, _, _ = result
 
        if isinstance(next_state, tuple):  # Handle tuple next_state
            next_state = next_state[0]
        next_state = np.reshape(next_state, [1, state_size])
 
        state = next_state
        total_reward += reward
 
        if done:  # If episode ends
            print(f"Evaluation Episode: {e+1}/{evaluation_episodes}, Score: {time}, Total Reward: {total_reward}")
            scores.append(total_reward)
            break
 
# Summary of evaluation performance
print(f"Average Reward: {np.mean(scores):.2f}, Max Reward: {np.max(scores)}, Min Reward: {np.min(scores)}")
 
env.close()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
Evaluation Episode: 1/10, Score: 7, Total Reward: 8.0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Evaluation Episode: 2/10, Score: 9, Total Reward: 10.0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━

**Result Interpretion**

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-2b7s{text-align:right;vertical-align:bottom}
.tg .tg-k9u1{border-color:inherit;color:#1B1C1D;font-size:100%;text-align:left;vertical-align:bottom}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-k9u1">Metric</th>
    <th class="tg-7zrl">Value</th>
    <th class="tg-7zrl">Interpretation</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-7zrl">Evaluation Episode</td>
    <td class="tg-2b7s">10/10</td>
    <td class="tg-0lax">The evaluation run completed successfully, testing the agent in 10 separate scenarios or attempts.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Score (Last Episode)</td>
    <td class="tg-2b7s">7</td>
    <td class="tg-0lax">The agent achieved a specific score of 7 in the very last evaluation episode (Episode 10). This score is specific to your environment's definition (e.g., 7 points, 7 steps survived, etc.).</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Total Reward (Last Episode)</td>
    <td class="tg-2b7s">8</td>
    <td class="tg-0lax">The agent accumulated a total reward of 8.0 in the very last episode.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Average Reward</td>
    <td class="tg-2b7s">14.6</td>
    <td class="tg-0lax">This is the most critical metric. On average, across all 10 evaluation episodes, the agent accumulated 14.60 units of reward. This is the best indicator of its typical performance.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Max Reward</td>
    <td class="tg-2b7s">65</td>
    <td class="tg-0lax">The agent achieved a reward of 65.0 in its single best run during the evaluation. This shows the agent's maximum potential capability, indicating it found a highly successful strategy at least once.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Min Reward</td>
    <td class="tg-2b7s">8</td>
    <td class="tg-0lax">The agent achieved a reward of 8.0 in its single worst run during the evaluation. This indicates the final episode was also the worst-performing episode in terms of reward.</td>
  </tr>
</tbody></table>

**Key Conclusion**

The large difference between the Average Reward ($14.60$), the Min Reward ($8.0$), and the Max Reward ($65.0$) suggests that the agent's performance is highly inconsistent (high variance). While it is capable of achieving great results ($65.0$), it often settles for much lower rewards ($8.0$) in typical or unsuccessful runs.

# Experiments

## Experiment 1: Explicit Reward Shaping for Stability

This experiment modifies the environment's Reward Function to include a shaping term that explicitly penalizes large pole angles, $r_{\text{shaped}} = r_{\text{original}} - \lambda \cdot \text{angle}$. By applying a negative reward (penalty) proportional to the absolute pole angle when the episode is active, the agent is incentivized to prioritize upright positions, promoting finer control and aiming to increase the average episode duration beyond the baseline

**Implementation**

The modified reward function penalizes large pole angles to incentivize upright pole positions.

In [9]:
def custom_reward(state, reward, done):
    """Modifies the reward to penalize large pole angles."""
    if not done:
        pole_angle = abs(state[2])  # Pole angle from state (index 2)
        angle_threshold = 0.1  # Approximately 5.7 degrees
        penalty = 0.5 * pole_angle if pole_angle > angle_threshold else 0  # Linear penalty
        return reward - penalty
    return reward  # Retain original reward (e.g., -10) for terminal states

**Integration into Training Loop**

Modify the training loop from the tutorial (Step 6) to incorporate the custom reward function. Below is the updated loop, assuming the tutorial’s setup.

**Expected Outcome**

- The penalty for large pole angles encourages the agent to prioritize upright pole positions, potentially increasing episode lengths.
- Expected scores may improve compared to the baseline (e.g., from ~10–50 to ~50–100), though convergence to ~200 may require more episodes or further tuning.
- Monitor printed episode scores to assess the impact.

**Notes**

- **Pole Angle:** The state’s third component (state[2]) represents the pole angle in radians. The threshold of 0.1 radians (~5.7°) targets significant deviations.
- **Penalty Scaling:** The penalty (0.5 * angle) is moderate to avoid overly discouraging exploration. Adjust the coefficient (e.g., 0.1 or 1.0) to experiment with penalty strength.
- **Compatibility:** The code integrates seamlessly with the tutorial’s setup, maintaining compatibility with both old and new Gym APIs.

In [10]:
# Experiment 1: Training loop with custom reward
episodes = 50
train_frequency = 5
target_update_frequency = 10

for e in range(episodes):
    state = env.reset()[0] if isinstance(env.reset(), tuple) else env.reset()
    state = np.reshape(state, [1, state_size])
    for time in range(200):
        action = act(state)
        step_result = env.step(action)
        next_state, reward, terminated, truncated, _ = step_result if len(step_result) == 5 else (*step_result, False)
        done = terminated or truncated
        reward = custom_reward(next_state, reward if not done else -10, done)  # Apply custom reward
        next_state = np.reshape(next_state, [1, state_size])
        remember(state, action, reward, next_state, done)
        state = next_state
        if done:
            print(f"Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.2f}")
            break
        if time % train_frequency == 0:
            replay(batch_size)
    if (e + 1) % target_update_frequency == 0:
        target_model.set_weights(model.get_weights())

Episode: 1/50, Score: 21, Epsilon: 0.19
Episode: 2/50, Score: 66, Epsilon: 0.17
Episode: 3/50, Score: 22, Epsilon: 0.17
Episode: 4/50, Score: 22, Epsilon: 0.17
Episode: 5/50, Score: 18, Epsilon: 0.16
Episode: 6/50, Score: 22, Epsilon: 0.16
Episode: 7/50, Score: 25, Epsilon: 0.15
Episode: 8/50, Score: 24, Epsilon: 0.15
Episode: 9/50, Score: 21, Epsilon: 0.15
Episode: 10/50, Score: 15, Epsilon: 0.14
Episode: 11/50, Score: 17, Epsilon: 0.14
Episode: 12/50, Score: 11, Epsilon: 0.14
Episode: 13/50, Score: 14, Epsilon: 0.14
Episode: 14/50, Score: 17, Epsilon: 0.13
Episode: 15/50, Score: 11, Epsilon: 0.13
Episode: 16/50, Score: 13, Epsilon: 0.13
Episode: 17/50, Score: 14, Epsilon: 0.13
Episode: 18/50, Score: 10, Epsilon: 0.13
Episode: 19/50, Score: 12, Epsilon: 0.13
Episode: 20/50, Score: 12, Epsilon: 0.12
Episode: 21/50, Score: 11, Epsilon: 0.12
Episode: 22/50, Score: 18, Epsilon: 0.12
Episode: 23/50, Score: 19, Epsilon: 0.12
Episode: 24/50, Score: 21, Epsilon: 0.11
Episode: 25/50, Score: 20

**Experiments results**

This experiment implemented Reward Shaping by adding a penalty proportional to the pole's angle. The theory is that this guides the agent to prioritize upright positions, leading to more stable, longer episodes.

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-k9u1{border-color:inherit;color:#1B1C1D;font-size:100%;text-align:left;vertical-align:bottom}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-k9u1">Metric</th>
    <th class="tg-7zrl">Observation</th>
    <th class="tg-7zrl">Interpretation</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-7zrl">Initial Performance (Ep 1-10)</td>
    <td class="tg-7zrl">Scores range from 15 to 66.</td>
    <td class="tg-0lax">Highly volatile performance, with one promising early spike (66).</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Mid-Training (Ep 11-40)</td>
    <td class="tg-7zrl">Scores stabilize but remain low (mostly 10 to 30).</td>
    <td class="tg-0lax">The agent struggles to maintain a policy and remains stuck in short-duration episodes.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Peak Performance</td>
    <td class="tg-7zrl">Score 85 in Episode 49.</td>
    <td class="tg-0lax">The agent finds a highly successful sequence late in training, but this success is not consistent.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Effectiveness</td>
    <td class="tg-7zrl">Low Convergence, High Variance.</td>
    <td class="tg-0lax">The shaping term did not yield the intended stable, long-term policy within 50 episodes. The agent remained highly inconsistent, suggesting the penalty may have been too disruptive or simply requires many more training episodes to be properly integrated into the Q-values.</td>
  </tr>
</tbody></table>

## Experiment 2: Early Stopping for Efficiency

This technique addresses training efficiency by implementing an early stopping criterion. Training is terminated once the agent demonstrates task mastery by consistently achieving an episode length of at least 195 steps over a predefined number of consecutive trials (e.g., 100). This mechanism prevents unnecessary computation once the agent's policy has converged to an optimal or near-optimal state.

**Implementation**

Add early stopping logic to the training loop using a list to track episode lengths.

**Expected Outcome**

- Early stopping triggers if the agent consistently achieves scores ≥195 for 100 episodes, indicating mastery of CartPole-v1.
- This reduces unnecessary training iterations, improving efficiency.
- If early stopping does not trigger within 50 episodes, consider increasing episodes (e.g., 500) or adjusting hyperparameters like learning rate.

**Notes**

- **Threshold:** The 195-step threshold is slightly below the maximum (200) to account for minor variations.
- **Tracking:** The episode_lengths list ensures accurate monitoring of consecutive successes.
- **Robustness:** The loop handles both terminating and non-terminating episodes, ensuring proper length recording.

In [11]:
# Early stopping parameters
consecutive_success_threshold = 100
success_episode_length = 195
episode_lengths = []

# Experiment 2: Training loop with early stopping
for e in range(episodes):
    state = env.reset()[0] if isinstance(env.reset(), tuple) else env.reset()
    state = np.reshape(state, [1, state_size])
    for time in range(200):
        action = act(state)
        step_result = env.step(action)
        next_state, reward, terminated, truncated, _ = step_result if len(step_result) == 5 else (*step_result, False)
        done = terminated or truncated
        reward = reward if not done else -10
        next_state = np.reshape(next_state, [1, state_size])
        remember(state, action, reward, next_state, done)
        state = next_state
        if done:
            episode_lengths.append(time)
            print(f"Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.2f}")
            # Early stopping check
            if (len(episode_lengths) >= consecutive_success_threshold and
                all(length >= success_episode_length for length in episode_lengths[-consecutive_success_threshold:])):
                print("Early stopping: Agent consistently achieves near-maximum episode length.")
                break
            break
        if time % train_frequency == 0:
            replay(batch_size)
    else:
        episode_lengths.append(time)  # Append max steps if episode doesn’t terminate
        print(f"Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.2f}")
    if (len(episode_lengths) >= consecutive_success_threshold and
        all(length >= success_episode_length for length in episode_lengths[-consecutive_success_threshold:])):
        print("Early stopping: Agent consistently achieves near-maximum episode length.")
        break
    if (e + 1) % target_update_frequency == 0:
        target_model.set_weights(model.get_weights())

Episode: 1/50, Score: 20, Epsilon: 0.05
Episode: 2/50, Score: 34, Epsilon: 0.05
Episode: 3/50, Score: 46, Epsilon: 0.05
Episode: 4/50, Score: 25, Epsilon: 0.05
Episode: 5/50, Score: 16, Epsilon: 0.05
Episode: 6/50, Score: 15, Epsilon: 0.05
Episode: 7/50, Score: 17, Epsilon: 0.05
Episode: 8/50, Score: 26, Epsilon: 0.04
Episode: 9/50, Score: 18, Epsilon: 0.04
Episode: 10/50, Score: 21, Epsilon: 0.04
Episode: 11/50, Score: 22, Epsilon: 0.04
Episode: 12/50, Score: 22, Epsilon: 0.04
Episode: 13/50, Score: 40, Epsilon: 0.04
Episode: 14/50, Score: 45, Epsilon: 0.04
Episode: 15/50, Score: 44, Epsilon: 0.04
Episode: 16/50, Score: 46, Epsilon: 0.03
Episode: 17/50, Score: 42, Epsilon: 0.03
Episode: 18/50, Score: 26, Epsilon: 0.03
Episode: 19/50, Score: 26, Epsilon: 0.03
Episode: 20/50, Score: 25, Epsilon: 0.03
Episode: 21/50, Score: 53, Epsilon: 0.03
Episode: 22/50, Score: 29, Epsilon: 0.03
Episode: 23/50, Score: 31, Epsilon: 0.03
Episode: 24/50, Score: 142, Epsilon: 0.02
Episode: 25/50, Score: 7

**Experiment results**

This experiment used the standard CartPole reward structure but introduced a practical mechanism: Early Stopping. The goal was to stop training once the agent consistently achieved mastery (score $\geq 195$).

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-k9u1{border-color:inherit;color:#1B1C1D;font-size:100%;text-align:left;vertical-align:bottom}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-k9u1">Metric</th>
    <th class="tg-7zrl">Observation</th>
    <th class="tg-7zrl">Interpretation</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-7zrl">Initial Performance (Ep 1-10)</td>
    <td class="tg-7zrl">Scores are low (15 to 46).</td>
    <td class="tg-0lax">Standard low performance typical of initial random exploration.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Convergence Point</td>
    <td class="tg-7zrl">Episode 33 hits score 199.</td>
    <td class="tg-0lax">This marks the point where the agent discovers the near-optimal policy.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Final Performance (Ep 43-50)</td>
    <td class="tg-7zrl">Consistent scores of 199 and 195.</td>
    <td class="tg-0lax">The agent rapidly masters the task. By the end of the 50 episodes, the policy is stable and near-perfect.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Effectiveness</td>
    <td class="tg-7zrl">High Convergence Speed.</td>
    <td class="tg-0lax">This run demonstrates that the default reward function and existing hyperparameters are highly effective. The agent achieved mastery within 33 episodes, proving the core model setup is robust. The early stopping mechanism itself (though not triggered due to the 50 episode limit) would have successfully saved computation time if the run were longer.</td>
  </tr>
</tbody></table>

## Experiment 3: Adaptive Greedy$\epsilon$-Decay Strategy

This refinement targets the Exploration-Exploitation Trade-off. It replaces the constant exponential decay with an adaptive schedule: a linear decay is used initially to ensure broad exploration, followed by a switch to exponential decay to rapidly shift the focus towards exploitation and fine-tune the optimal policy. This hybrid strategy is designed to accelerate convergence by optimizing the timing of exploration.

**Implementation**

Update the epsilon decay logic and integrate it into the training loop.

**Expected Outcome**

- Linear decay in the first 25 episodes promotes broad exploration, while exponential decay afterward shifts to exploitation.
- Scores may improve faster than the baseline (constant 0.995 decay) due to prolonged early exploration, potentially reaching ~50–100 within 50 episodes.
- Compare average scores over the last 10 episodes to the baseline.

**Notes**

- **Switch Point:** Episode 25 balances sufficient exploration with timely exploitation. Adjust to 50 for more exploration.
- **Decay Rates:** Linear decay (0.01) ensures steady exploration reduction; exponential decay (0.995) aligns with the tutorial’s baseline.
- **Integration:** Epsilon updates occur after replay and at episode end to maintain consistency.

In [12]:
def decay_epsilon(epsilon, episode, switch_episode=25):
    """Applies linear decay before switch_episode, then exponential decay."""
    if episode < switch_episode:
        return max(epsilon - 0.01, epsilon_min)  # Linear decay
    return max(epsilon * epsilon_decay, epsilon_min)  # Exponential decay

# Experiment 3: Training loop with adaptive epsilon decay
epsilon = 1.0
for e in range(episodes):
    state = env.reset()[0] if isinstance(env.reset(), tuple) else env.reset()
    state = np.reshape(state, [1, state_size])
    for time in range(200):
        action = act(state)
        step_result = env.step(action)
        next_state, reward, terminated, truncated, _ = step_result if len(step_result) == 5 else (*step_result, False)
        done = terminated or truncated
        reward = reward if not done else -10
        next_state = np.reshape(next_state, [1, state_size])
        remember(state, action, reward, next_state, done)
        state = next_state
        if done:
            print(f"Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.2f}")
            break
        if time % train_frequency == 0:
            replay(batch_size)
            epsilon = decay_epsilon(epsilon, e)  # Update epsilon after replay
    else:
        print(f"Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.2f}")
    if (e + 1) % target_update_frequency == 0:
        target_model.set_weights(model.get_weights())
    epsilon = decay_epsilon(epsilon, e)  # Update epsilon at episode end

Episode: 1/50, Score: 11, Epsilon: 0.96
Episode: 2/50, Score: 68, Epsilon: 0.75
Episode: 3/50, Score: 117, Epsilon: 0.43
Episode: 4/50, Score: 199, Epsilon: 0.01
Episode: 5/50, Score: 199, Epsilon: 0.01
Episode: 6/50, Score: 199, Epsilon: 0.01
Episode: 7/50, Score: 199, Epsilon: 0.01
Episode: 8/50, Score: 199, Epsilon: 0.01
Episode: 9/50, Score: 199, Epsilon: 0.01
Episode: 10/50, Score: 199, Epsilon: 0.01
Episode: 11/50, Score: 199, Epsilon: 0.01
Episode: 12/50, Score: 199, Epsilon: 0.01
Episode: 13/50, Score: 199, Epsilon: 0.01
Episode: 14/50, Score: 199, Epsilon: 0.01
Episode: 15/50, Score: 199, Epsilon: 0.01
Episode: 16/50, Score: 199, Epsilon: 0.01
Episode: 17/50, Score: 199, Epsilon: 0.01
Episode: 18/50, Score: 199, Epsilon: 0.01
Episode: 19/50, Score: 199, Epsilon: 0.01
Episode: 20/50, Score: 199, Epsilon: 0.01
Episode: 21/50, Score: 199, Epsilon: 0.01
Episode: 22/50, Score: 199, Epsilon: 0.01
Episode: 23/50, Score: 199, Epsilon: 0.01
Episode: 24/50, Score: 199, Epsilon: 0.01
Epi

**Experiment result**

This experiment modified the crucial Exploration-Exploitation Trade-off by applying linear $\epsilon$-decay early on for broad exploration, then switching to exponential decay for fast exploitation.

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-k9u1{border-color:inherit;color:#1B1C1D;font-size:100%;text-align:left;vertical-align:bottom}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-k9u1">Metric</th>
    <th class="tg-7zrl">Observation</th>
    <th class="tg-7zrl">Interpretation</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-7zrl">Convergence Speed</td>
    <td class="tg-7zrl">Achieved Score 199 by Episode 4.</td>
    <td class="tg-0lax">This is an exceptionally fast rate of convergence.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Policy Stability</td>
    <td class="tg-7zrl">Consistent score of 199 from Episode 4 onward.</td>
    <td class="tg-0lax">The agent achieves and maintains a perfect policy for the remaining 47 episodes.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Epsilon Decay</td>
    <td class="tg-7zrl">ϵ drops rapidly to 0.01 by Episode 4.</td>
    <td class="tg-0lax">The adaptive strategy successfully forced rapid convergence once the agent found the optimal region of the state space.</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Effectiveness</td>
    <td class="tg-7zrl">Extremely High Success and Efficiency.</td>
    <td class="tg-0lax">This strategy was the most successful. The prolonged, steady exploration in the first few episodes allowed the agent to gather enough critical data. The subsequent fast decay locked the agent into exploitation mode quickly, accelerating learning dramatically compared to the other two methods.</td>
  </tr>
</tbody></table>

## Experiments Summary

These experiments enhance the DQN setup by:

- **Reward Shaping:** Penalizing large pole angles to encourage stability.
- **Early Stopping:** Stop training upon consistent high performance for efficiency.
- **Adaptive Epsilon Decay:** Switching from linear to exponential decay for balanced exploration-exploitation.

Each experiment is concise, integrates with the tutorial’s code, and encourages analysis of performance impacts through printed scores. The code avoids dependencies on irrelevant components (e.g., ImageDataGenerator from the original exercises) and focuses on reinforcement learning concepts.